In [1]:
import uuid
import logging
from base64 import b64encode
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
load_dotenv()

True

## Evaluation pipeline

### Prepare data

In [2]:
from evals import Dataset

dataset_name = "THRD-2021-163881"
dc = Dataset(dataset_name)
data = dc.load_dataset()

if not data or "sessions" not in data:
    print("Ingen sessions funnet")
    exit()

dc.assign_session_attachments()

total_attachments = sum(len(s.get("attachments", [])) for s in data["sessions"])
logger.info(f"Done — {len(data['sessions'])} sessions, {total_attachments} attachments assigned in total")

INFO:evals.dataset:Found 120 files under datasets/THRD-2021-163881/01_data/
INFO:evals.dataset:Session Prosjekt-initialisering | – → 2020-03-15 | 18 candidates, 18 new
INFO:evals.dataset:Session Forbehold om heving | 2020-03-15 → 2020-05-20 | 7 candidates, 7 new
INFO:evals.dataset:Session Formell heving | 2020-05-20 → 2021-06-25 | 7 candidates, 7 new
INFO:evals.dataset:Session Stevning og tilsvar | 2021-06-25 → 2022-01-15 | 5 candidates, 5 new
INFO:evals.dataset:Session Forberedelse rettsmekling | 2022-01-15 → 2022-03-10 | 2 candidates, 2 new
INFO:evals.dataset:Session Forliksavtale | 2022-03-10 → 2022-04-05 | 4 candidates, 4 new
INFO:evals.dataset:Session Forliksbrudd | 2022-04-05 → 2023-04-20 | 24 candidates, 24 new
INFO:evals.dataset:Session Gjenopptakelse | 2023-04-20 → 2023-07-10 | 7 candidates, 7 new
INFO:evals.dataset:Session Prosesskriv og sluttinnlegg | 2023-07-10 → 2025-06-05 | 23 candidates, 23 new
INFO:evals.dataset:Session Dom | 2025-06-05 → 2025-07-20 | 20 candidates, 20 

### GATHER RESULTS

In [3]:
from agent.agent import Agent
from models import AskAgentRequest, AttachmentModel
from agent.utils import PROMPT
from agent.tools import TOOLS
import os
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg_pool import AsyncConnectionPool

class CollectAgentResult:
    def __init__(self, data: dict, llm_model: str = "google_gemini-2.5-pro"):
        self.data = data
        self.llm_model = llm_model
        self.dataclass = Dataset(name=data.get("dataset_name", "default_dataset"))

    async def init_agent(self):
        connection_string = os.getenv("SUPABASE_DB_URL")
        pool = AsyncConnectionPool(conninfo=connection_string, open=False)
        await pool.open()
        checkpointer = AsyncPostgresSaver(pool)
        agent = Agent(
            tools=TOOLS,
            prompt=PROMPT,
            checkpointer=checkpointer,
        )
        logger.info("Agent initialized with AsyncPostgresSaver checkpointer")
        return agent

    def file_type_map(self, blob_path: str):
        mapping = {
            "txt": "text/plain",
            "pdf": "application/pdf",
            "docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
            "xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            "csv": "text/csv",
            "md": "text/markdown",
            "eml": "message/rfc822",
        }
        return mapping.get(blob_path.split(".")[-1], None)

    def parse_attachments(self, attachment_path: str, query_id: str, session_id: str, user_id: str):
        bytes_content = self.dataclass.bucket.blob(attachment_path).download_as_bytes()
        content = b64encode(bytes_content).decode("utf-8")
        file_id = str(uuid.uuid4())
        return AttachmentModel(
            filename=attachment_path,
            file_type=self.file_type_map(attachment_path),
            content=content,
            file_id=file_id,
            path=f"{user_id}/{session_id}/{file_id}.{attachment_path.split('.')[-1]}",
            size=len(content),
            query_id=query_id,
        )

    async def run_conv(self, conv, agent_class, project_id, session_id, query_id, user_id):
        input_obj = AskAgentRequest(
            question=conv.get("input"),
            session_id=session_id,
            llm_model=self.llm_model,
            query_id=query_id,
            project_id=project_id,
            attachments=[],
        )
        answer = "No content"
        async for response in agent_class.stream_response(query=input_obj, user_id=user_id):
            if response.get("type") == "ai":
                answer = response.get("data", {}).get("token_stream", "No content")
        conv["model_response"] = answer

    async def run_agent(self):
        agent_class = await self.init_agent()
        logger.info("=========== STARTING EVALUATION ===========")
        logger.info(
            f'Dataset: {self.data.get("dataset_name")} | '
            f'Sessions: {len(self.data.get("sessions", []))} | '
            f'Project: {self.data.get("project_id")} | '
            f'User: {self.data.get("user_id")}'
        )
        self.data["llm_model"] = self.llm_model

        for idx, session in enumerate(self.data.get("sessions", [])):
            logger.info(
                f"Session {idx} | {session.get('date')} | "
                f"{session.get('session_name')} | "
                f"{len(session.get('attachments', []))} attachments"
            )
            query_id = str(uuid.uuid4())
            attachments = [
                self.parse_attachments(
                    attachment_path=att,
                    query_id=query_id,
                    session_id=session.get("session_id", "unknown_session"),
                    user_id=self.data.get("user_id", "unknown_user"),
                )
                for att in session.get("attachments", [])
            ]

            input_obj = AskAgentRequest(
                question=session.get("init_query"),
                session_id=session.get("session_id"),
                llm_model=self.llm_model,
                query_id=query_id,
                project_id=self.data.get("project_id"),
                attachments=attachments,
            )

            if idx == 0:
                async for response in agent_class.initialize_project(
                    query=input_obj, user_id=self.data.get("user_id")
                ):
                    logger.debug(f"Init response: {response}")
            else:
                async for response in agent_class.update_project(
                    query=input_obj, user_id=self.data.get("user_id")
                ):
                    logger.debug(f"Update response: {response}")

            for conv in session["conversation"]:
                await self.run_conv(
                    conv=conv,
                    agent_class=agent_class,
                    project_id=self.data.get("project_id"),
                    session_id=session.get("session_id"),
                    query_id=query_id,
                    user_id=self.data.get("user_id"),
                )

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


In [ ]:
car = CollectAgentResult(data, llm_model="google_gemini-2.5-flash")
await car.run_agent()

In [ ]:
# llm_model = "google_gemini-2.5-pro"
# logger.info(f"=========== STARTING EVALUATION ===========")
# logger.info(f'Dataset name : {dataset_name} | Total sessions: {len(data.get("sessions", []))} | Project ID: {data.get("project_id", "Unknown")} | User ID: {data.get("user_id", "Unknown")}')
# logger.info(f'===========================================')


# data["llm_model"] = llm_model
# agent_class = await init_agent()
# for idx, session in enumerate(data.get("sessions", [])):
#     if idx < 1: #FOR TESTING
#         #====================
#         # INIT OR UPDATE
#         #====================
#         logger.info(f"Session date: {session['date']}, session name: {session['session_name']} attachments: {len(session['attachments'])}")
#         attachments = []
#         query_id = str(uuid.uuid4())
#         for i, att in enumerate(session["attachments"]):
#             #if i < 3: #FOR TESTING
#             bytes_content = data.bucket.blob(att).download_as_bytes()
#             content = b64encode(bytes_content).decode('utf-8')
#             file_id = str(uuid.uuid4())
#             attachment_obj = AttachmentModel(
#                 filename=Path(att).name,
#                 file_type = file_type_map.get(Path(att).suffix.lstrip('.'), None),
#                 content=content,
#                 file_id = file_id,
#                 path = f"{data.get('user_id')}/{session['session_id']}/{file_id}.{Path(att).suffix.lstrip('.')}",
#                 size = len(content),
#                 query_id = query_id
#             )
#             attachments.append(attachment_obj)

#         input_obj = AskAgentRequest(
#                                     question=session.get("init_query"),
#                                     session_id = session.get("session_id"),
#                                     llm_model=llm_model,
#                                     query_id = query_id,
#                                     project_id=data.get("project_id"),
#                                     attachments=attachments
#                                     )
#         if idx == 0:
#             async for response in agent_class.initialize_project(query = input_obj,
#                                                     user_id = data.get("user_id"),
#                                                     ):
#                 logger.debug(f"Received response during initialization: {response}")
#         else:
#             async for response in agent_class.update_project(query=input_obj,
#                                                         user_id = data.get("user_id"),):
#                     logger.debug(f"Received response during update: {response}")
        
#         #====================
#         # RUN CONVERSATION
#         #====================
        
#         for conv in session["conversation"]:
#             input_obj = AskAgentRequest(
#                                         question=conv.get("input"),
#                                         session_id = session.get("session_id"),
#                                         llm_model="google_gemini-2.5-pro",
#                                         query_id = query_id,
#                                         project_id=data.get("project_id"),
#                                         attachments=[],
                                        
#                                         )
#             async for response in agent_class.stream_response(query = input_obj,
#                                                     user_id = data.get("user_id"),
#                                                     ):
#                 if response.get("type") == "ai":
#                     ai_response = response
#                     answer = response.get("data", {}).get("token_stream", "No content")
#                     logger.info(f"Received answer: {answer}")
            
#             conv["model_response"] = answer


INFO:__main__:=========== STARTING EVALUATION ===========
INFO:__main__:Dataset name : THRD-2021-163881 | Total sessions: 10 | Project ID: ee9ee007-92f7-4fdc-ba5a-d7cb55694241 | User ID: 53d63d18-cfa1-416e-96e8-770c8f66507b
INFO:__main__:===========================================
INFO:__main__:Agent initialized with AsyncPostgresSaver checkpointer
INFO:__main__:Session date: 2020-03-15, session name: Prosjekt-initialisering attachments: 18
DEBUG:__main__:Received response during initialization: {'type': 'status', 'phase': ['parse-documents'], 'status': 'starting', 'data': {'attachments': 18}, 'timestamp': '2026-02-24T12:22:39.060416', 'query_id': '611c000b-1e32-42cc-9cea-8eeecec3ab10'}
DEBUG:__main__:Received response during initialization: {'type': 'status', 'phase': ['parse_doc'], 'status': 'starting', 'data': {'filename': '1994-08-15_37_byggetillatelse_1994_hovedbygg.txt', 'file_id': '0f790ae3-7fd9-4105-b92f-0a3eef8a32d3', 'progress': 0, 'total': 18}, 'timestamp': '2026-02-24T12:22

Storage endpoint URL should have a trailing slash.


INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/0f790ae3-7fd9-4105-b92f-0a3eef8a32d3.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/2d315ba2-c672-4d27-8774-8e9ad36ca961.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/dddce0d0-fede-4efd-9b3a-70c462a3c533.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/d13f490e-a205-4922-903c-295c4a34711c.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1/dc75f5b2-f524-4918-a839-3db76cc34c06.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63

In [49]:
dc.save_results(data)

INFO:__main__:Results saved to datasets/THRD-2021-163881/04_results/google_gemini-2.5-pro_2026-02-24_12-37-01.json


### EVALUATE RESULTS

In [7]:
import deepeval